# Phase 0: Create ICD and RxNorm Translation Dictionaries

## Purpose
This notebook creates translation dictionaries that convert medical codes into human-readable descriptions:
- **ICD codes** (diagnosis and procedure codes) → Medical descriptions via UMLS API
- **RxNorm CUI codes** → Medication ingredient names via RxNorm API
- **NDC codes** → Medication ingredient names via RxNorm/NDC API

## Inputs
- Raw PCDM tables from `input_data/Data_import.py`:
  - `PCDM_DIAGNOSIS` - Contains ICD diagnosis codes
  - `PCDM_PROCEDURES` - Contains procedure codes (CPT, ICD)
  - `PCDM_MED_ADMIN` - Contains medication administration codes (RxNorm, NDC)
  - `PCDM_PRESCRIBING` - Contains prescription codes (RxNorm CUI)

## Outputs
Translation CSV files saved to `input_data/ICD_RX translation/`:
- `HNC Diagnosis ICD Translation DF.csv` - ICD diagnosis code translations
- `HNC procedure ICD Translation DF.csv` - Procedure code translations
- `rxCodeTranslationDF_HNC.csv` - RxNorm CUI to medication name
- `ndcCodeTranslationDF_HNC.csv` - NDC code to medication name

## Key Functions
- `query_UMLS(query, codingSystem)` - Queries UMLS API for ICD code translations
- `translateICD(query)` - Tries multiple coding systems to translate ICD codes
- `extractIngredientFromRxNorm(query)` - Extracts active ingredient from RxNorm
- `NDCtoRXNorm(query)` - Converts NDC codes to RxNorm CUI codes
- `queryNDC(query)` - Queries NDC code via RxNorm API

## Execution Time
- Approximately 20-30 minutes depending on dataset size and API response times

## When to Run
- **Initial setup only** (one-time execution)
- When new codes appear in the dataset that aren't in existing translation files
- To update translations if API definitions change

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
import math
import os


### Import data

In [ ]:
from input_data import Data_import

In [ ]:
### Import data
PCDM_DIAGNOSIS = Data_import.PCDM_DIAGNOSIS
PCDM_MED_ADMIN = Data_import.PCDM_MED_ADMIN
PCDM_PROCEDURES = Data_import.PCDM_PROCEDURES
PCDM_DEMOGRAPHIC = Data_import.PCDM_DEMOGRAPHIC
PCDM_PRESCRIBING = Data_import.PCDM_PRESCRIBING

In [ ]:
### Pull procedures through ICD translation
def query_UMLS(query, codingSystem):
    query = str.upper(query)
    codingSystem = str.upper(codingSystem)
    base_url = 'https://uts-ws.nlm.nih.gov/rest'
    apikey = os.getenv('UMLS_API_KEY') ### Set your UMLS API key as an environment variable
    endpoint = f'content/current/source/{codingSystem}/{query}?'
    apiQuery = f'{base_url}/{endpoint}apiKey={apikey}'
    #print(apiQuery)
    resp = requests.get(apiQuery)
    return resp.json()['result']['name']                                                                                                                         

codingSystems = ['CPT', 'HCPT', 'HCPCS', 'OT', 'ICD9CM', 'ICD10CM']

def translateICD(query):
    codingSystems = ['CPT', 'HCPT', 'HCPCS', 'OT', 'ICD9CM', 'ICD10CM']
    descript = np.nan
    for code in codingSystems:
        try:
            descript = query_UMLS(str(query), code)
            break
        except:
            continue
    return descript

In [ ]:
def queryRXNorm(query):
    query = str(query)
    base_url = 'https://rxnav.nlm.nih.gov/REST/rxcui/'
    endpoint = f'{query}.json'
    apiQuery = f'{base_url}{endpoint}'
    resp = requests.get(apiQuery)
    return resp.json()['idGroup']['name']

def queryRXNormAllRelated(query):
    query = str(query)
    base_url = 'https://rxnav.nlm.nih.gov/REST/rxcui/'
    endpoint = f'{query}/allrelated.json'
    apiQuery = f'{base_url}{endpoint}'
    resp = requests.get(apiQuery)
    return resp.json()['allRelatedGroup']['conceptGroup']

def queryNDC(query):
    query = str(query)
    base_url = 'https://rxnav.nlm.nih.gov/REST/ndcstatus.json?'
    endpoint = f'ndc={query}'
    apiQuery=f'{base_url}{endpoint}'
    resp = requests.get(apiQuery)
    return resp.json()['ndcStatus']['rxcui']

def queryRXNormAllRelatedNDC(query):
    query = str(query)
    rxnormQuery = queryNDC(query)
    return queryRXNormAllRelated(rxnormQuery)

def extractIngredientFromRxNorm(query):
    query = str(query)
    base_url = 'https://rxnav.nlm.nih.gov/REST/rxcui/'
    endpoint = f'{query}/allrelated.json'
    apiQuery = f'{base_url}{endpoint}'
    resp = requests.get(apiQuery)
    conceptGroups = resp.json()['allRelatedGroup']['conceptGroup']
    output = ''
    for val in conceptGroups:
        if val['tty'] == 'IN':
            output = val['conceptProperties'][0]['name']
    return output

def NDCtoRXNorm(query):
    query = str(query)
    base_url = 'https://rxnav.nlm.nih.gov/REST/ndcstatus.json?'
    endpoint = f'ndc={query}'
    apiQuery=f'{base_url}{endpoint}'
    resp = requests.get(apiQuery)
    return resp.json()['ndcStatus']['rxcui']

### Translate med admin/Prescribing codes

In [ ]:
PCDM_MED_ADMIN.dropna(axis=0, subset='MEDADMIN_CODE', inplace=True)

In [ ]:
PCDM_PRESCRIBING.dropna(axis=0,subset='RXNORM_CUI', inplace=True)

In [ ]:
rxcodesToTranslate = list(set(  list(set(PCDM_MED_ADMIN[PCDM_MED_ADMIN['MEDADMIN_TYPE']=='RX']['MEDADMIN_CODE']))  + list(set(PCDM_PRESCRIBING['RXNORM_CUI']))  ))
ndcCodesToTranslate = list(set(PCDM_MED_ADMIN[PCDM_MED_ADMIN['MEDADMIN_TYPE']=='ND']['MEDADMIN_CODE']))

In [ ]:
### rxcode translation
rxCodeTranslateDict = {}
for code in tqdm(rxcodesToTranslate):
    outcome = ''
    try:
        try:
            outcome = extractIngredientFromRxNorm(int(code))
        except:
            outcome = extractIngredientFromRxNorm(code)
    except:
        outcome = ''
    
    rxCodeTranslateDict[code]= outcome

In [ ]:
### ndc translation
ndcTranslateDict = {}
for code in tqdm(ndcCodesToTranslate):
    outcome = ''
    try:
        try: 
            outcome = extractIngredientFromRxNorm(int(code))
        except:
            outcome = extractIngredientFromRxNorm(code)
    except:
        try:
            try: 
                rxnormCode = NDCtoRXNorm(int(code))
                outcome = extractIngredientFromRxNorm(rxnormCode)
            except:
                rxnormCode = NDCtoRXNorm(code)
                outcome = extractIngredientFromRxNorm(rxnormCode)
        except:
            outcome = ''
    
    ndcTranslateDict[code]= outcome

In [ ]:
rxCodeTranslatedf = pd.DataFrame.from_dict(rxCodeTranslateDict, orient = 'index', columns = [ 'TRANSLATION'])
rxCodeTranslatedf  = rxCodeTranslatedf.reset_index()
rxCodeTranslatedf.columns = ['RX_CODE', 'TRANSLATION']
rxCodeTranslatedf.to_csv('input_data/ICD_RX translation/rxCodeTranslationDF_HNC.csv')

In [ ]:
ndcTranslatedf = pd.DataFrame.from_dict(ndcTranslateDict, orient = 'index', columns=['TRANSLATION'])
ndcTranslatedf  = ndcTranslatedf.reset_index()
ndcTranslatedf.columns = ['NDC_CODE', 'TRANSLATION']
ndcTranslatedf.to_csv('input_data/ICD_RX translation/ndcCodeTranslationDF_HNC.csv')

### Translate diagnosis and procedures

In [ ]:
### Pull procedures through ICD translation
def query_UMLS(query, codingSystem):
    query = str.upper(query)
    codingSystem = str.upper(codingSystem)
    base_url = 'https://uts-ws.nlm.nih.gov/rest'
    apikey = os.getenv('UMLS_API_KEY') ### Set your UMLS API key as an environment variable for security
    endpoint = f'content/current/source/{codingSystem}/{query}?'
    apiQuery = f'{base_url}/{endpoint}apiKey={apikey}'
    #print(apiQuery)
    resp = requests.get(apiQuery)
    return resp.json()['result']['name']                                                                                                                         

codingSystems = ['CPT', 'HCPT', 'HCPCS', 'OT', 'ICD9CM', 'ICD10CM']

def translateICD(query):
    codingSystems = ['CPT', 'HCPT', 'HCPCS', 'OT', 'ICD9CM', 'ICD10CM']
    descript = np.nan
    for code in codingSystems:
        try:
            descript = query_UMLS(str(query), code)
            break
        except:
            continue
    return descript

In [ ]:
PCDM_DIAGNOSIS

In [ ]:
translateDict = {}
diagnosisList = list(set(PCDM_DIAGNOSIS['DX']))
for code in tqdm(diagnosisList):
    translatedCode = translateICD(code)
    translateDict[code] = translatedCode

In [ ]:
### Manual review
#### Should be corrected after investigating weird files
code = 'S34.109A'
translateDict[code] = translateICD(code)
translateDict[code]

In [ ]:
proceduretranslateDict = {}
proceduresList = list(set(PCDM_PROCEDURES['PX']))
for code in tqdm(proceduresList):
    translatedCode = translateICD(code)
    proceduretranslateDict[code] = translatedCode

In [ ]:
### Save for later use
procedCodeTranslationDF = pd.DataFrame.from_dict(proceduretranslateDict, orient = 'index',columns=['DESCRIPTION']).reset_index(names='ICD_CODES')
procedCodeTranslationDF.to_csv('input_data/ICD_RX translation/HNC procedure ICD translation DF.csv')

In [ ]:
###Save this for later use
DiagCodeTranslationDF = pd.DataFrame.from_dict(translateDict, orient='index',columns=['DESCRIPTION']).reset_index(names='ICD_CODES')
DiagCodeTranslationDF.to_csv('input_data/ICD_RX translation/HNC Diagnosis ICD Translation DF.csv')